# Explicit module reloading with `reloadm`

This notebook builds a tiny package, edits a child module, and repairs the package-level re-export.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

from reloadm import plan, reload

workspace = tempfile.TemporaryDirectory()
root = Path(workspace.name)
package = root / "demo_shop"
package.mkdir()
(package / "__init__.py").write_text("from .pricing import rate\n", encoding="utf-8")
(package / "pricing.py").write_text("rate = 10\n", encoding="utf-8")
sys.path.insert(0, str(root))

In [ ]:
import demo_shop
import demo_shop.pricing

assert demo_shop.rate == 10
plan("demo_shop.pricing", include_parents=True).modules

In [ ]:
(package / "pricing.py").write_text("rate = 222\n", encoding="utf-8")
importlib.invalidate_caches()
reload(demo_shop.pricing, include_parents=True)

assert demo_shop.pricing.rate == 222
assert demo_shop.rate == 222
demo_shop.rate

In [ ]:
sys.path.remove(str(root))
for name in ("demo_shop.pricing", "demo_shop"):
    sys.modules.pop(name, None)
workspace.cleanup()